# Classical ML

In [29]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB  # Import Naive Bayes classifier

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)  # Forward fill as an example

# Categorize AMS scores into binary classes
def categorize_ams(score):
    return 'AMS' if score >= 3 else 'NO AMS'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)']]
y = df['AMS Encoded']

# Dynamic oversampling for balanced classes
df_no_ams = df[df['AMS Category'] == 'NO AMS']
df_ams = df[df['AMS Category'] == 'AMS']

max_size = max(len(df_no_ams), len(df_ams))

df_no_ams_upsampled = resample(df_no_ams, replace=True, n_samples=max_size, random_state=40)
df_ams_upsampled = resample(df_ams, replace=True, n_samples=max_size, random_state=40)

df_combined = pd.concat([df_no_ams_upsampled, df_ams_upsampled])
X_resampled = df_combined[['HR (bpm)', 'SpO2 (%)']]
y_resampled = df_combined['AMS Encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=40)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model definitions
models = {
    'Random Forest': RandomForestClassifier(n_estimators=2, max_depth=1, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=2, random_state=42),  # Default base_estimator is DecisionTreeClassifier
    'Logistic Regression': LogisticRegression(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=2, random_state=42),
    'SVM': SVC(kernel='linear', probability=True),
    'Naive Bayes': GaussianNB(),
}

# Function to estimate model memory usage
def calculate_memory(model):
    if hasattr(model, 'estimators_'):  # For ensemble models like RandomForest and Bagging
        memory = sum(estimator.tree_.node_count * 4 for estimator in model.estimators_ if hasattr(estimator, 'tree_')) / (1024 * 1024)
    elif hasattr(model, 'support_vectors_'):  # For SVM
        memory = model.support_vectors_.nbytes / (1024 * 1024)
    else:
        memory = 0  # For other models, we assume negligible memory
    return memory

# Train and evaluate each model
results = {}

for name, model in models.items():
    # Train the model
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    end_time = time.time()

    # Predictions
    predictions_train = model.predict(X_train_scaled)
    predictions_test = model.predict(X_test_scaled)

    # Evaluation
    train_acc = accuracy_score(y_train, predictions_train) * 100
    test_acc = accuracy_score(y_test, predictions_test) * 100
    memory_required = calculate_memory(model)

    # Store results
    results[name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Training Time (seconds)': end_time - start_time,
        'Model Memory (MB)': memory_required
    }

# Print results for each model
for model_name, metrics in results.items():
    print(f"\n{model_name} Results:")
    print(f"Train Accuracy: {metrics['Train Accuracy']:.2f}%")
    print(f"Test Accuracy: {metrics['Test Accuracy']:.2f}%")
    print(f"Training Time: {metrics['Training Time (seconds)']:.4f} seconds")
    print(f"Model Memory Required: {metrics['Model Memory (MB)']:.4f} MB")

    # Display classification report for each model
    predictions_test = models[model_name].predict(X_test_scaled)
    print("\nClassification Report:")
    print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))


Random Forest Results:
Train Accuracy: 64.77%
Test Accuracy: 50.00%
Training Time: 0.0050 seconds
Model Memory Required: 0.0000 MB

Classification Report:
              precision    recall  f1-score   support

         AMS       0.48      0.89      0.63        18
      NO AMS       0.60      0.15      0.24        20

    accuracy                           0.50        38
   macro avg       0.54      0.52      0.43        38
weighted avg       0.55      0.50      0.42        38


Bagging Results:
Train Accuracy: 88.64%
Test Accuracy: 73.68%
Training Time: 0.0035 seconds
Model Memory Required: 0.0002 MB

Classification Report:
              precision    recall  f1-score   support

         AMS       0.65      0.94      0.77        18
      NO AMS       0.92      0.55      0.69        20

    accuracy                           0.74        38
   macro avg       0.79      0.75      0.73        38
weighted avg       0.79      0.74      0.73        38


Logistic Regression Results:
Train Accu

# HDC

# Pseudo random

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils import resample
from sobol_seq import i4_sobol_generate
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)

# Categorize AMS scores into binary classes
def categorize_ams(score):
    return 'AMS' if score >= 3 else 'NO AMS'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Feature Engineering: Additional features
df['HR_SpO2_Ratio'] = df['HR (bpm)'] / (df['SpO2 (%)'] + 1e-6)
df['SpO2_HR_Ratio'] = df['SpO2 (%)'] / (df['HR (bpm)'] + 1e-6)
df['HR_Squared'] = df['HR (bpm)'] ** 2
df['SpO2_Squared'] = df['SpO2 (%)'] ** 2

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y = df['AMS Encoded']

# Use SMOTE for balanced classes
smote = SMOTE(random_state=40)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Select best features
selector = SelectKBest(f_classif, k=5)  # You can change k based on your analysis
X_selected = selector.fit_transform(X_resampled, y_resampled)

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_resampled, test_size=0.3, random_state=40, stratify=y_resampled
)

# Normalize features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hyperdimensional Computing (HDC) parameters
D = 50  # Experiment with higher dimensionality
NUM_CLASSES = len(np.unique(y_train))
NUM_SAMPLES = X_train_scaled.shape[0]

# Create a Sobol sequence projection matrix
proj = i4_sobol_generate(X_train_scaled.shape[1], D)

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train_scaled, proj)

# Create class hypervectors
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train.iloc[i]] += X_train_proj[i]

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("HDC (Cosine Similarity) Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test_scaled, proj)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print("HDC (Cosine Similarity) Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + class_hypervectors.nbytes

print("HDC (Cosine Similarity) Training Time: {:.4f} seconds".format(training_time))
print("HDC (Cosine Similarity) Inference Time: {:.4f} seconds".format(inference_time))
print("HDC (Cosine Similarity) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

# Display classification report
print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))

HDC (Cosine Similarity) Train Accuracy:  60.22727272727273
HDC (Cosine Similarity) Test Accuracy:  68.42105263157895
HDC (Cosine Similarity) Training Time: 0.0105 seconds
HDC (Cosine Similarity) Inference Time: 0.0017 seconds
HDC (Cosine Similarity) Model Memory Required: 0.0027 MB
              precision    recall  f1-score   support

         AMS       0.73      0.58      0.65        19
      NO AMS       0.65      0.79      0.71        19

    accuracy                           0.68        38
   macro avg       0.69      0.68      0.68        38
weighted avg       0.69      0.68      0.68        38



# Quasi random

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, PolynomialFeatures
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils import resample
from sobol_seq import i4_sobol_generate
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif

# Load the data
df = pd.read_excel('ams_data.xlsx')

# Check for missing values
if df.isnull().sum().any():
    df.fillna(method='ffill', inplace=True)

# Categorize AMS scores into binary classes
def categorize_ams(score):
    return 'AMS' if score >= 3 else 'NO AMS'

df['AMS Category'] = df['AMS Total Score'].apply(categorize_ams)

# Encode categories
label_encoder = LabelEncoder()
df['AMS Encoded'] = label_encoder.fit_transform(df['AMS Category'])

# Feature Engineering: Additional features
df['HR_SpO2_Ratio'] = df['HR (bpm)'] / (df['SpO2 (%)'] + 1e-6)
df['SpO2_HR_Ratio'] = df['SpO2 (%)'] / (df['HR (bpm)'] + 1e-6)
df['HR_Squared'] = df['HR (bpm)'] ** 2
df['SpO2_Squared'] = df['SpO2 (%)'] ** 2

# Prepare features and labels
X = df[['HR (bpm)', 'SpO2 (%)', 'HR_SpO2_Ratio', 'SpO2_HR_Ratio', 'HR_Squared', 'SpO2_Squared']]
y = df['AMS Encoded']

# Use SMOTE for balanced classes
smote = SMOTE(random_state=40)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Select best features
selector = SelectKBest(f_classif, k=5)  # You can change k based on your analysis
X_selected = selector.fit_transform(X_resampled, y_resampled)

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_resampled, test_size=0.3, random_state=40, stratify=y_resampled
)

# Normalize features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hyperdimensional Computing (HDC) parameters
D = 50  # Experiment with different dimensionalities
NUM_CLASSES = len(np.unique(y_train))
NUM_SAMPLES = X_train_scaled.shape[0]

# Create a Sobol sequence projection matrix
proj = i4_sobol_generate(X_train_scaled.shape[1], D)

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train_scaled, proj)

# Create class hypervectors
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train.iloc[i]] += X_train_proj[i]

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("HDC (Cosine Similarity) Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test_scaled, proj)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print("HDC (Cosine Similarity) Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = proj.nbytes + class_hypervectors.nbytes

print("HDC (Cosine Similarity) Training Time: {:.4f} seconds".format(training_time))
print("HDC (Cosine Similarity) Inference Time: {:.4f} seconds".format(inference_time))
print("HDC (Cosine Similarity) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

# Display classification report
print(classification_report(y_test, predictions_test, target_names=label_encoder.classes_))


HDC (Cosine Similarity) Train Accuracy:  60.22727272727273
HDC (Cosine Similarity) Test Accuracy:  68.42105263157895
HDC (Cosine Similarity) Training Time: 0.0100 seconds
HDC (Cosine Similarity) Inference Time: 0.0009 seconds
HDC (Cosine Similarity) Model Memory Required: 0.0027 MB
              precision    recall  f1-score   support

         AMS       0.73      0.58      0.65        19
      NO AMS       0.65      0.79      0.71        19

    accuracy                           0.68        38
   macro avg       0.69      0.68      0.68        38
weighted avg       0.69      0.68      0.68        38

